# Quick Eval: Base Qwen 7B vs Direct GRPO-Trained Model

Head-to-head comparison judged by Llama 70B. Generates responses from both models on the same prompts, randomizes A/B order, and tallies wins.

## 1. Setup

In [1]:
!pip install -q torch transformers datasets peft accelerate openai gdown

In [2]:
import os
os.environ["TOGETHER_API_KEY"] = "d38edcb211441a6dc2884be10f8ed63d38fc4181f5adbe66f841a0b9a98d2ede"  # <-- replace this

In [3]:
import json
import random

import openai
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA: True
GPU: NVIDIA A100-SXM4-80GB


In [4]:
import os
import gdown

# Download the adapter folder.
gdown.download_folder(url="https://drive.google.com/drive/folders/1SEnYiA0HlFTQMYUVbkWwJBTtoXtQo7G0?usp=drive_link", output=".", quiet=False)

# Some gdown runs skip nested files; fetch preferences directly if needed.
if not os.path.exists("./grpo_direct/preferences.jsonl"):
    gdown.download(
        url="https://drive.google.com/file/d/17s8ETX9-7Irt_Ocoh7A8_Ncy6foGfy5Y/view?usp=drive_link",
        output="./grpo_direct/preferences.jsonl",
        fuzzy=True,
        quiet=False,
    )

assert os.path.exists("./grpo_direct/preferences.jsonl"), "preferences.jsonl missing from grpo_direct"
!ls -la grpo_direct && echo "---" && ls grpo_direct/

Downloading...
From: https://drive.google.com/uc?id=17s8ETX9-7Irt_Ocoh7A8_Ncy6foGfy5Y
To: /content/preferences.jsonl
100%|██████████| 58.0M/58.0M [00:01<00:00, 50.8MB/s]
Retrieving folder contents


Retrieving folder 1Jgv1srKjZVhr-Fn5SXWgXwQK9XqiiFTr checkpoint-50
Processing file 1FAmL2nM3HdESG-Ba9u0qbV9Pa837a1Y1 adapter_config.json
Processing file 1wAsLjk7v8mRL5XimVhlI_uHvJ4vQ-IF9 adapter_model.safetensors
Processing file 1hkkgOj_cysPy2UiiZGHIdsdRVh4Z1011 chat_template.jinja
Processing file 1AblG2Heoco3UbFSZQ2LHi4y46eUkp_sb optimizer.pt
Processing file 1DVXHDKhHDVvKz1by5q_9s0Hz21beiwNZ README.md
Processing file 1N_xfnymJUPvGED4BJamtmdZv090dTote rng_state.pth
Processing file 1PKRDcE8KXvV6tI9iQw5GPgYNV3hDgdrm scheduler.pt
Processing file 1RM-NPzMAEVjtQCkYYjR7i08xPAc4LjmU tokenizer_config.json
Processing file 1XSPgXZNVXMHrTYKI63rm8Kx1Bm9rR56x tokenizer.json
Processing file 1651zwizFuJmxSKou1LZs8DIZcRPlAFFu trainer_state.json
Processing file 1b3Uv3WhL4Hpp8hwmpsYCWCSHj2YV2Ud- training_args.bin
Retrieving folder 1yf-EnAYUR70YLijWWM1k18nJSXPDKMco checkpoint-100
Processing file 1mO4W6FGUnwO-uFQnrws9ldTkvuZcOsco adapter_config.json
Processing file 1XTQLe4Dg8CeYMZlLiJ_o0POeS5AGC2Ov adapter

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1FAmL2nM3HdESG-Ba9u0qbV9Pa837a1Y1
To: /content/grpo_direct/checkpoint-50/adapter_config.json
100%|██████████| 1.01k/1.01k [00:00<00:00, 2.98MB/s]
Downloading...
From: https://drive.google.com/uc?id=1wAsLjk7v8mRL5XimVhlI_uHvJ4vQ-IF9
To: /content/grpo_direct/checkpoint-50/adapter_model.safetensors
100%|██████████| 40.4M/40.4M [00:00<00:00, 45.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1hkkgOj_cysPy2UiiZGHIdsdRVh4Z1011
To: /content/grpo_direct/checkpoint-50/chat_template.jinja
100%|██████████| 2.51k/2.51k [00:00<00:00, 8.10MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1AblG2Heoco3UbFSZQ2LHi4y46eUkp_sb
From (redirected): https://drive.google.com/uc?id=1AblG2Heoco3UbFSZQ2LHi4y46eUkp_sb&confirm=t&uuid=c5c1bcea-0e25-446c-a3cd-327d6b6117c3
To: /content/grpo_direct/checkpoint-50/optimizer.pt
100%|█████

-rw-r--r-- 1 root root 57989208 Mar  9 00:51 preferences.jsonl
---
adapter_config.json	   checkpoint-200     tokenizer_config.json
adapter_model.safetensors  checkpoint-50      tokenizer.json
chat_template.jinja	   eval_quick.ipynb   training_args.bin
checkpoint-100		   preferences.jsonl
checkpoint-150		   README.md



Download completed


## 2. Download data from Google Drive

In [5]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_PATH = "./grpo_direct"          # path to your saved LoRA adapter
DATA_PATH = "./grpo_direct/preferences.jsonl"
JUDGE_MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

NUM_EVAL = 20                            # number of prompts to evaluate
MAX_NEW_TOKENS = 256

client = openai.OpenAI(
    api_key=os.environ["TOGETHER_API_KEY"],
    base_url="https://api.together.xyz/v1",
)

## 3. Load both models

Base model + LoRA adapter loaded on GPU. Base model alone used for comparison.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Load base model
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=dtype, device_map="auto",
)
base_model.eval()

# Load trained model (base + LoRA adapter)
print("Loading trained model (base + LoRA)...")
trained_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
trained_model.eval()
print("Both models loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loading trained model (base + LoRA)...
Both models loaded.


## 4. Helper functions

In [7]:
JUDGE_TEMPLATE = """You are an impartial judge. Given an instruction and two responses, decide which response is better.

Consider: accuracy, helpfulness, clarity, and conciseness.

Respond with EXACTLY "A" or "B" (no other text).

Instruction: {instruction}

Response A:
{response_a}

Response B:
{response_b}"""


def generate_local(model, instruction: str) -> str:
    """Generate a response from a local model."""
    messages = [{"role": "user", "content": instruction}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7, do_sample=True, top_p=0.9,
        )
    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def judge(instruction: str, response_a: str, response_b: str) -> str:
    """Ask Llama 70B which response is better. Returns 'A', 'B', or 'TIE'."""
    prompt = JUDGE_TEMPLATE.format(
        instruction=instruction, response_a=response_a, response_b=response_b,
    )
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=0.0,
    )
    text = resp.choices[0].message.content.strip().upper()
    if text in ("A", "B"):
        return text
    return "TIE"


def get_eval_prompts(path: str, n: int) -> list[str]:
    """Get n random unique prompts from preferences.jsonl."""
    prompts = []
    seen = set()
    with open(path) as f:
        for line in f:
            row = json.loads(line)
            inst = row["instruction"]
            if inst not in seen:
                seen.add(inst)
                prompts.append(inst)
    random.seed(123)
    return random.sample(prompts, min(n, len(prompts)))

## 5. Run evaluation

In [8]:
prompts = get_eval_prompts(DATA_PATH, NUM_EVAL)
print(f"Evaluating {len(prompts)} prompts\n")

trained_wins = 0
base_wins = 0
ties = 0
results = []

for i, instruction in enumerate(prompts):
    print(f"[{i+1}/{len(prompts)}] {instruction[:80]}...")

    # Generate from both models
    # Disable adapter for base model response
    trained_model.disable_adapter_layers()
    base_response = generate_local(trained_model, instruction)

    # Enable adapter for trained model response
    trained_model.enable_adapter_layers()
    trained_response = generate_local(trained_model, instruction)

    # Randomize A/B order for judge
    swap = random.random() < 0.5
    if swap:
        v = judge(instruction, trained_response, base_response)
        winner = "trained" if v == "A" else "base" if v == "B" else "tie"
    else:
        v = judge(instruction, base_response, trained_response)
        winner = "base" if v == "A" else "trained" if v == "B" else "tie"

    if winner == "trained":
        trained_wins += 1
    elif winner == "base":
        base_wins += 1
    else:
        ties += 1

    results.append({
        "instruction": instruction,
        "base_response": base_response,
        "trained_response": trained_response,
        "winner": winner,
    })

    print(f"  Winner: {winner}")
    print(f"  Base:    {base_response[:120]}...")
    print(f"  Trained: {trained_response[:120]}...")
    print()

Evaluating 20 prompts

[1/20] What is the five-letter English word that means "to criticize severely" and can ...
  Winner: trained
  Base:    The five-letter English word that means "to criticize severely" and can be formed by rearranging the letters in "crouste...
  Trained: The five-letter word that means "to criticize severely" and can be formed by rearranging the letters in "croustet" is "s...

[2/20] In what ways can the Latex data format be utilized to accurately and comprehensi...
  Winner: trained
  Base:    To utilize LaTeX to define the concept of "vegetarianism" within a Python program, we can leverage LaTeX's ability to pr...
  Trained: LaTeX is primarily used for typesetting documents and isn't directly suitable for defining concepts programmatically in ...

[3/20] Can you solve this puzzle? 
Find the exact area of a rectangle that has a width ...
  Winner: trained
  Base:    Thank you for the challenge! Let's break down the problem step-by-step using the given dimensions

## 6. Results

In [10]:
print("=" * 60)
print(f"RESULTS ({len(prompts)} prompts)")
print(f"  Trained model wins: {trained_wins}")
print(f"  Base model wins:    {base_wins}")
print(f"  Ties:               {ties}")
print(f"  Trained win rate:   {trained_wins / len(prompts):.1%}")
print(f"  Base win rate:      {base_wins / len(prompts):.1%}")
print("=" * 60)

# Save results
with open("eval_results.jsonl", "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
print(f"\nDetailed results saved to eval_results.jsonl")

RESULTS (20 prompts)
  Trained model wins: 16
  Base model wins:    4
  Ties:               0
  Trained win rate:   80.0%
  Base win rate:      20.0%

Detailed results saved to eval_results.jsonl
